In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from typing import TypedDict, Annotated
import os 
os.environ["HF_HOME"] = "E:\\huggingface_embedding_cache"
from langchain_huggingface import HuggingFaceEmbeddings



e:\learning AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

config = {
    "configurable":{
        "thread_id":"000001"
    }
}
store = InMemoryStore(index={
    "dims": 1536,
    "embed": model
})
saver = InMemorySaver()
class State(TypedDict):
    question: str

namespace = ('user', 'nadeem')

def add_memory(state: State, store: BaseStore):
   store.put(namespace, 
          'favourite Sport',
          {"data": "Nadeem likes to play cricket"}
        )
store.put(namespace, 
          'hobby',
          {"data": "Nadeems hobby is gym"}
        )
store.put(namespace, 
          'food',
          {"data": "Nadeem likes to eat eggs"}
        )
store.put(namespace, 
          'today',
          {"data": "Nadeem will watch football final today"}
        )


def fetch_memories(state: State, store: BaseStore):
    all_memories = store.search(('user', 'nadeem'), query='what is user doing today', limit=3)
    for memory in all_memories:
        print(memory)
    return {}


graph = StateGraph(State)


graph.add_node('chat_node', add_memory)
graph.add_node('expl_node', fetch_memories)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', 'expl_node')
graph.add_edge('expl_node', END)

workflow = graph.compile(checkpointer=saver, store=store)

result = workflow.invoke({'question':""},
                         config=config)
# print(result)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3894.43it/s]


Item(namespace=['user', 'nadeem'], key='today', value={'data': 'Nadeem will watch football final today'}, created_at='2026-07-19T06:42:07.004344+00:00', updated_at='2026-07-19T06:42:07.004349+00:00', score=0.15857705989885557)
Item(namespace=['user', 'nadeem'], key='hobby', value={'data': 'Nadeems hobby is gym'}, created_at='2026-07-19T06:42:06.948534+00:00', updated_at='2026-07-19T06:42:06.948540+00:00', score=0.10123439591641399)
Item(namespace=['user', 'nadeem'], key='favourite Sport', value={'data': 'Nadeem likes to play cricket'}, created_at='2026-07-19T06:42:07.061284+00:00', updated_at='2026-07-19T06:42:07.061288+00:00', score=0.0909450363426126)
